# Pandas 3: GROUP BY and Business Analysis

[https://pandas.pydata.org/](https://pandas.pydata.org/)

This notebook continues the teaching sequence from the reference Pandas notebook and then uses the same skills to solve the Office Solutions business problem.

### Our learning plan for Notebook 3
- use `groupby()`
- calculate means, sums, and counts by group
- sort grouped results
- group by more than one column
- use `.agg()` for multiple summary functions
- work with dates using `.dt`
- combine Pandas skills to investigate the Tables profitability problem
- make a data-supported business recommendation

In [ ]:
import pandas as pd
sales = pd.read_csv("TableauSalesData.csv")
print(sales.head())

## GROUP BY in Pandas

`groupby()` lets us split rows into groups and calculate a summary for each group.

A useful way to recognize a `groupby()` question is to look for the word **by**:

- average Profit **by Region**
- total Sales **by Category**
- average Discount **by Segment**

Documentation: [https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)

In [ ]:
sales.groupby('Region')

Calling `groupby()` by itself creates a grouped object. To get a useful result, select a column (or columns) and apply an **aggregate function** such as `.mean()` or `.sum()`.

In [ ]:
# average profit by region
sales.groupby('Region')['Profit'].mean()

### Exercise 1

1. Calculate the average `Sales` for each `Segment`.
2. Calculate the average `Sales` and `Profit` for each `Region`.

## 1.average Sales for each Segment


In [ ]:
sales.groupby("Segment")["Sales"].mean()

## 2.average Sales and Profit for each Region


In [ ]:
sales.groupby("Region")[["Sales", "Profit"]].mean()

## Using `.sum()` with `groupby()`

For business performance questions, totals are often more meaningful than row-level averages.

In [ ]:
# total Sales and Profit by Category
sales.groupby("Category")[["Sales", "Profit"]].sum()

### Exercise 2

Calculate total `Sales` and `Profit` for each `Sub-Category`.

## Your code is here


In [ ]:
sales.groupby("Sub-Category")[["Sales", "Profit"]].sum()

## Sorting grouped results

Once we calculate a summary, `.sort_values()` helps us rank the results and quickly identify the **highest** or **lowest** performers.

The basic syntax is:

```python
dataframe.sort_values(by="column_name")
```

By default, Pandas sorts from **smallest to largest**.

Documentation: [https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)


In [ ]:
# First, create a summary table
subcat_performance = sales.groupby("Sub-Category")[["Sales", "Profit"]].sum()

subcat_performance.sort_values(
    by="Profit",
    ascending=False
)

### Sort from lowest to highest

`ascending=True` means **smallest → largest**.

This is the default behavior, so these two lines produce the same result:

```python
subcat_performance.sort_values(by="Profit")
subcat_performance.sort_values(by="Profit", ascending=True)
```

This is useful when we want to find the **largest losses** first.


In [ ]:
# lowest Profit to highest Profit
subcat_performance[subcat_performance["Profit"] < 0].sort_values(
    by="Profit"
)


### Sort from highest to lowest with `ascending=False`

Use `ascending=False` when you want to reverse the order:

```python
dataframe.sort_values(by="column_name", ascending=False)
```

So:

- `ascending=True` → lowest to highest
- `ascending=False` → highest to lowest

For example, if we want to identify the most profitable sub-categories first, sort `Profit` in descending order.


In [ ]:
# highest Profit to lowest Profit
subcat_performance.sort_values(by="Profit", ascending=False)


### Business question: top performers

Suppose a manager asks:

> Which sub-categories generate the most total profit?

We can combine `groupby()`, `.sum()`, and `.sort_values()` in one statement.


In [ ]:
sales.groupby("Sub-Category")[["Sales", "Profit"]].sum().sort_values(
    by="Profit",
    ascending=False
)


### Sorting by a different column

The column inside `by=` determines **what variable we rank**.

For example, we can rank sub-categories by total `Sales` instead of `Profit`.


In [ ]:
# highest Sales to lowest Sales
subcat_performance.sort_values(by="Sales", ascending=False)


### Quick practice

Create total `Sales` and `Profit` by `Sub-Category`, then sort the Sub-Category from **highest total Profit to lowest total Profit**.

**Hint:** use `ascending=False`.


## Your code is here


In [ ]:
subcat_performance.sort_values(by="Profit", ascending=False)

### Stop and interpret subcat_performance

Which sub-categories have negative total profit? Which one has the largest loss?

Do **not** immediately recommend discontinuing a sub-category. A good analyst should drill down to understand *where* and *why* the losses occur.

## Grouping by more than one column

We can pass a list of columns to `groupby()` when we want a more detailed breakdown.

In [ ]:
# total Sales and Profit by Category and Sub-Category
sales.groupby(["Category", "Sub-Category"])[["Sales", "Profit"]].sum()


### Exercise 3

Calculate total `Sales` and `Profit` by:

1. `Region` and `Segment`
2. `Region` and `Sub-Category`

## Your code is here

In [ ]:
## Total Sales and Profit by Region and Segment
sales.groupby(["Region", "Segment"])[["Sales", "Profit"]].sum()

## Total Sales and Profit by Region and Sub-Category
sales.groupby(["Region", "Sub-Category"])[["Sales", "Profit"]].sum()

# Business Investigation: Why are Tables losing money?

We will now combine the Pandas skills from all three notebooks. Start by selecting only the `Tables` sub-category.

In [ ]:
tables = sales.loc[sales["Sub-Category"] == "Tables"].copy()

tables.head()

## Tables by customer segment

In [ ]:
tables.groupby("Segment")[["Sales", "Profit"]].sum().sort_values(by="Profit")

### Exercise 4

Calculate total Tables `Sales` and `Profit` by `Region`. Sort the result from lowest to highest Profit.

## Your code is here


In [ ]:
tables.groupby("Region")[["Sales", "Profit"]].sum().sort_values(by="Profit", ascending=True)

## Combining filtering and grouping

Suppose we discover that the West performs differently. We can first filter to the West, then group by Segment.

In [ ]:
west_tables = tables.loc[tables["Region"] == "West"]

west_tables.groupby("Segment")[["Sales", "Profit"]].sum().sort_values(by="Profit")

### Exercise 5

For Tables in the **Central** region, calculate total `Sales` and `Profit` by `Segment`.

Which segment, if any, is profitable?

In [ ]:
## Your code is here
central_tables = tables.loc[tables["Region"] == "Central"]

central_profit = central_tables.groupby("Segment")[["Sales", "Profit"]].sum()

central_profit

central_profit[central_profit["Profit"] > 0]

## Average discounts

If discounts are too large, they may contribute to low profitability. Let's calculate average discount by region.

In [ ]:
tables.groupby("Region")["Discount"].mean().sort_values(ascending=False)

## Using `.agg()` to apply different functions

Sometimes we want total Sales and Profit **and** average Discount in the same table. `.agg()` lets us apply different summary functions.

Documentation: [https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html)

In [ ]:
tables_by_region = tables.groupby("Region").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Avg_Discount=("Discount", "mean")
)

tables_by_region.sort_values("Profit")

### Exercise 6

Use `.agg()` to create a table by `Segment` with:

- total Sales
- total Profit
- average Discount
- total Quantity

Sort from lowest to highest Profit.

In [ ]:
# Your code is here
tables.groupby("Segment").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Avg_Discount=("Discount", "mean"),
    Quantity=("Quantity", "sum")
).sort_values(by="Profit", ascending=True)

## Dates and the `.dt` accessor

Pandas provides the `.dt` accessor for datetime columns. We can extract year, month, quarter, and other date components.

Documentation: [https://pandas.pydata.org/docs/user_guide/basics.html#dt-accessor](https://pandas.pydata.org/docs/user_guide/basics.html#dt-accessor)

In [ ]:
# Convert Order Date to datetime
sales["Order Date"] = pd.to_datetime(sales["Order Date"])

# Create a Year column
sales["Year"] = sales["Order Date"].dt.year

# Recreate Tables with the Year column
tables = sales.loc[sales["Sub-Category"] == "Tables"].copy()

# Total Tables Sales and Profit by Year
yearly_tables = tables.groupby("Year")[["Sales", "Profit"]].sum()

yearly_tables

In [ ]:
# recreate Tables so it includes the Year column
tables = sales.loc[sales["Sub-Category"] == "Tables"].copy()

yearly_tables = tables.groupby("Year")[["Sales", "Profit"]].sum()
yearly_tables

### Exercise 7

Create a `Quarter` column from `Order Date` using `.dt.quarter`. Then calculate total Tables `Sales` and `Profit` by quarter.

In [ ]:
# Your code is here

sales["Quarter"] = sales["Order Date"].dt.quarter

tables = sales.loc[sales["Sub-Category"] == "Tables"].copy()

tables.groupby("Quarter")[["Sales", "Profit"]].sum()

## Resetting the index

After `groupby()`, the grouping variable often becomes the index. `.reset_index()` turns it back into a normal column, which is especially convenient for plotting.

Documentation: [https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html)

In [ ]:
yearly_tables_plot = yearly_tables.reset_index()
yearly_tables_plot

## A simple Pandas visualization

In [ ]:
ax = yearly_tables_plot.plot(
    x="Year",
    y="Sales",
    kind="bar",
    title="Office Solutions: Table Sales by Year",
    legend=False
)
ax.set_ylabel("Sales ($)")

## Final Business Challenge

Natasha wants a recommendation about the Tables sub-category. Based on your analysis, choose one of the following positions, or propose another data-supported action:

- discontinue Tables entirely
- continue Tables but reduce discounts
- keep Tables only in selected regions/segments
- another recommendation supported by the data

### Your task
Use **at least three pieces of evidence**. Your evidence should come from Pandas analyses you can reproduce in code.

Suggested questions to investigate:

1. How large is the total Tables loss?
2. Which regions and segments perform best/worst?
3. Are there profitable combinations of Region and Segment?
4. What do average discounts look like where profits are worst?
5. How have Tables Sales and Profit changed over time?

In [34]:
# Evidence 1
# Your code is here

tables[["Sales", "Profit"]].sum()

Sales     206965.5320
Profit    -17725.4811
dtype: float64

In [ ]:
# Evidence 2
# Your code is here

tables.groupby(["Region", "Segment"])[["Sales", "Profit"]].sum().sort_values("Profit")

In [ ]:
# Evidence 3
# Your code is here

tables.groupby("Region").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Avg_Discount=("Discount", "mean")
).sort_values("Profit")

**Recommendation:**  
I recommend keeping Tables but lowering discounts to reduce losses.

**Evidence 1:**  
Tables made $206,965 in sales but lost $17,725.

**Evidence 2:**  
East Corporate had the biggest loss at $5,299.66, while West Corporate made $1,074.20 in profit.

**Evidence 3:**  
The East had the highest average discount at 37.4% and lost $11,025.38. The West had a 20% average discount and made $1,482.61 in profit.

**Business reasoning:**  
I think we should keep Tables because some areas are making money. We should lower discounts in areas like the East and see if that helps reduce losses.

## Check your mastery

By the end of these three notebooks, you should be able to decide which Pandas tool fits a business question:

- inspect data → `.head()`, `.shape`, `.columns`, `.dtypes`
- summarize data → `.describe()`, `.value_counts()`, `.unique()`
- select columns → `[]` and `[[]]`
- select rows → Boolean conditions and `.loc[]`
- answer a question containing **by** → `groupby()`
- total values → `.sum()`
- average values → `.mean()`
- rank results → `.sort_values()`; use `ascending=False` for highest-to-lowest
- combine summary functions → `.agg()`
- analyze time → `.dt.year`, `.dt.quarter`

The key goal is not to memorize syntax. It is to learn how to translate a **business question** into a sequence of Pandas operations.
